# BeeWare AI Capstone Notebook

This notebook builds a CNN-based audio classifier from scratch using MFCC features and TensorFlow.

In [ ]:
!pip install librosa tensorflow pandas scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [ ]:
df = pd.read_csv('all_data_updated.csv')
df.head()

In [ ]:
DATA_PATH = 'all_data'
X = []
y = []

for _, row in df.iterrows():
    file_path = os.path.join(DATA_PATH, row['filename'])
    signal, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40)
    mfcc = mfcc[:, :130]
    if mfcc.shape[1] < 130:
        pad_width = 130 - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)))
    X.append(mfcc)
    y.append(str(row['label']))

X = np.array(X)
X = X.reshape(X.shape[0], X.shape[1], X.shape[2], 1)

In [ ]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_categorical,
    test_size=0.30,
    random_state=42,
    stratify=y_categorical,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(40, 130, 1)),
    MaxPooling2D(),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(y_categorical.shape[1], activation='softmax'),
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
)

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print('Test accuracy:', accuracy)

In [ ]:
model.save('model/beeware_model.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('model/beeware_model.tflite', 'wb') as f:
    f.write(tflite_model)

with open('model/labels.txt', 'w', encoding='utf-8') as f:
    for label in encoder.classes_:
        f.write(f'{label}\n')